In [1]:
import time
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

In [2]:
DATASET_NAME = "CLAP 0.5s"

DATASET_PATH = (
    "/Users/bhavaykhatri/Desktop/msclap_2023/"
    "singBAP_dataset_clap-2023_0.5s.parquet"
)

df = pd.read_parquet(DATASET_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Shape: (34409, 13)
Columns: ['condition', 'experience', 'extractor', 'filename', 'filepath', 'frame_index', 'phonation', 'scale', 'singer', 'take', 'embedding', 'embedding_dtype', 'embedding_shape']


,condition,experience,extractor,filename,filepath,frame_index,phonation,scale,singer,take,embedding,embedding_dtype,embedding_shape
0,after_instruction,inexperienced,clap-2023,inex-1-after_instruction-glissando-1-mic-audio,/home/suvihaara/Documents/PhD/DATA/VTE/SingBAP...,0,,glissando,INEX-1,1,b'\x97!\xa9\xbf\xb9\x86\x80?\x8d\xa1t?z\xa51>\...,<f4,[1024]
1,after_instruction,inexperienced,clap-2023,inex-1-after_instruction-glissando-1-mic-audio,/home/suvihaara/Documents/PhD/DATA/VTE/SingBAP...,1,,glissando,INEX-1,1,b't\xf4\xae?\xd1\xcd\xaf\xbeVGM?\xdc\x94\x94\x...,<f4,[1024]
2,after_instruction,inexperienced,clap-2023,inex-1-after_instruction-glissando-1-mic-audio,/home/suvihaara/Documents/PhD/DATA/VTE/SingBAP...,2,,glissando,INEX-1,1,b'\xb1\xe0\x85?\'\xe1E\xbfBv-?\x89~!\xbd\xd1\x...,<f4,[1024]
3,after_instruction,inexperienced,clap-2023,inex-1-after_instruction-glissando-1-mic-audio,/home/suvihaara/Documents/PhD/DATA/VTE/SingBAP...,3,,glissando,INEX-1,1,b'\xaeU\xda\xbe\xdet\x99\xbe.8\xca>=\xf8b\xbd+...,<f4,[1024]
4,after_instruction,inexperienced,clap-2023,inex-1-after_instruction-glissando-1-mic-audio,/home/suvihaara/Documents/PhD/DATA/VTE/SingBAP...,4,,glissando,INEX-1,1,b'\x0c\xc8\x08?\xceB \xbf\xdcc\x8a?x\x1b\xd2=)...,<f4,[1024]


In [3]:
TARGET_CLASSES = [
    "correct",
    "arched_back",
    "hunched_back",
    "sideways",
    "chest_breathing",
    "over_articulation",
    "under_articulation",
]

df = df[
    df["experience"].isin(
        ["intermediate", "professional"]
    )
].copy()

df = df[
    df["condition"].isin(TARGET_CLASSES)
].copy()

print("Filtered shape:", df.shape)
print(df["condition"].value_counts())
print(df["experience"].value_counts())

Filtered shape: (28418, 13)
condition
correct               5137
hunched_back          4540
sideways              4285
chest_breathing       4259
over_articulation     3454
under_articulation    3381
arched_back           3362
Name: count, dtype: int64
experience
intermediate    23420
professional     4998
Name: count, dtype: int64


In [4]:
def decode_embedding(value):
    if isinstance(
        value,
        (bytes, bytearray, memoryview),
    ):
        return np.frombuffer(
            value,
            dtype=np.float32,
        ).copy()

    return np.asarray(
        value,
        dtype=np.float32,
    ).reshape(-1)

In [5]:
X = np.vstack(
    df["embedding"].apply(decode_embedding)
)

y = df["condition"].astype(str).to_numpy()

groups = df["filename"].astype(str).to_numpy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Unique recordings:", len(np.unique(groups)))

X shape: (28418, 1024)
y shape: (28418,)
Unique recordings: 3046


In [6]:
splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(
        X,
        y,
        groups=groups,
    )
)

X_train = X[train_idx]
X_test = X[test_idx]

y_train = y[train_idx]
y_test = y[test_idx]

train_groups = groups[train_idx]
test_groups = groups[test_idx]

print("Train:", X_train.shape)
print("Test:", X_test.shape)

print(
    "Shared recordings:",
    len(
        set(train_groups)
        & set(test_groups)
    ),
)

Train: (22735, 1024)
Test: (5683, 1024)
Shared recordings: 0


In [7]:
MODELS = {
    "MLP": make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=(256, 128),
            early_stopping=True,
            max_iter=300,
            random_state=42,
        ),
    ),

    "KNN": make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(
            n_neighbors=15,
            metric="cosine",
            n_jobs=-1,
        ),
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),

    "Linear SVM": make_pipeline(
        StandardScaler(),
        LinearSVC(
            class_weight="balanced",
            max_iter=10000,
            random_state=42,
        ),
    ),
}

In [8]:
results = []

for model_name, base_model in MODELS.items():
    print(f"Training {model_name}...")

    model = clone(base_model)

    start = time.time()

    model.fit(
        X_train,
        y_train,
    )

    predictions = model.predict(
        X_test
    )

    elapsed = time.time() - start

    results.append({
        "Embedding": DATASET_NAME,
        "Model": model_name,

        "Accuracy": round(
            accuracy_score(
                y_test,
                predictions,
            ),
            4,
        ),

        "Balanced Accuracy": round(
            balanced_accuracy_score(
                y_test,
                predictions,
            ),
            4,
        ),

        "Macro F1": round(
            f1_score(
                y_test,
                predictions,
                average="macro",
                zero_division=0,
            ),
            4,
        ),

        "Train/Eval Time (s)": round(
            elapsed,
            2,
        ),

        "Samples": len(y),
        "Features": X.shape[1],
    })

Training MLP...
Training KNN...
Training Random Forest...
Training Linear SVM...


In [9]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "Macro F1",
    ascending=False,
).reset_index(drop=True)

results_df

,Embedding,Model,Accuracy,Balanced Accuracy,Macro F1,Train/Eval Time (s),Samples,Features
0,CLAP 0.5s,MLP,0.3762,0.3828,0.3790,16.86,28418,1024
1,CLAP 0.5s,Linear SVM,0.3217,0.3366,0.3173,388.86,28418,1024
2,CLAP 0.5s,Random Forest,0.2998,0.3099,0.2970,25.11,28418,1024
3,CLAP 0.5s,KNN,0.2914,0.2914,0.2938,2.18,28418,1024


In [10]:
results_df.to_csv(
    "clap_0.5s_baseline.csv",
    index=False,
)

print("Saved: clap_0.5s_baseline.csv")

Saved: clap_0.5s_baseline.csv
